#### Assignment Objectives:
<ol>
  <li>Produce JSON files.</li>
  <li>Each race from races.csv.</l1>
  <li>One file per year.</li>
  <li>Naming: stats_{year}.json. --> ./results </li>
  <li>Date and time handling (all in UTC, if time not available --> 00:00:00)</li>
  <li>The winner --> position 1 for the respective race.</li>
  <li>If the key is always a number --> put the right data type.</li>
  <li>Should be in a main.py</li>
  <li>Clear doc how to run the pipeline from a python terminal.</li>
  <li>Mention any stretch requirements that have been undertaken.</li>
  <li>Create req.txt to run for all imports</li>
  <li>Take into consideration that for a british race maybe we need the time in their timezone instead of UTC</li>
</ol>

In [ ]:
import pandas as pd
import json

In [ ]:
races = pd.read_csv('../source-data/races.csv')
results = pd.read_csv('../source-data/results.csv')

In [17]:
results.head()

,resultId,raceId,driverId,position,fastestLapTime
0,26505,1132,1,1.0,01:29.4
1,26506,1132,830,2.0,01:29.0
2,26507,1132,846,3.0,01:29.3
3,26508,1132,857,4.0,01:28.7
4,26509,1132,832,5.0,01:28.3


In [18]:
# can have more results for one single race --> parent (races) - child (results)
joined_tables = races.merge(results, on='raceId', how='left')

In [19]:
# check for duplicates after merging
duplicates_check = joined_tables[joined_tables.duplicated(subset=['raceId', 'driverId'], keep=False)]
assert duplicates_check.empty == True

#### date time management

In [20]:
# would usually add a Z at the end as it is UTC
# when exporting the time: mention explicitly the format as: joined_tables['datetime'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

joined_tables['time'].fillna('00:00:00', inplace=True)
joined_tables['datetime'] = pd.to_datetime(joined_tables['date'] + ' ' + joined_tables['time'])
joined_tables['datetime'] = joined_tables['datetime'].dt.strftime('%Y-%m-%dT%H:%M:%S.000')

### Winner per race

In [21]:
# making sure our position col is the right type -- float
# joined_tables['position'] = pd.to_numeric(joined_tables['position'], errors='coerce')
winner_driver = joined_tables[joined_tables['position'] == 1.0][['raceId', 'driverId']] # keeping the raceid to remerge the tables

In [22]:
winner_driver = winner_driver.rename(columns={'driverId': "Race Winning driverId"})

In [23]:
winner_driver

,raceId,Race Winning driverId
12,1132,1.0
32,1131,847.0
52,1130,830.0
72,1129,830.0
92,1128,844.0
...,...,...
2651,993,1.0
2671,992,1.0
2691,991,817.0
2711,990,20.0


#### handle nans/missing values

In [24]:
# checks
print(f'There are {joined_tables["driverId"].isna().sum()} races that have no race results')

There are 12 races that have no race results


In [25]:
def from_fastest_lap_time_to_seconds(fastest_lap_time):
    if pd.isna(fastest_lap_time):
        return None
    minutes, seconds = fastest_lap_time.split(':')
    total_in_seconds = float(minutes) * 60 + float(seconds)
    
    return total_in_seconds

In [26]:
joined_tables['fastestLapTime'] = joined_tables['fastestLapTime'].apply(from_fastest_lap_time_to_seconds)

### Fastest lap per race

### in this case we keep the nans for now so we dont lose records, altenatively, we could drop them

In [27]:
fastest_lap_per_race = joined_tables.groupby('raceId')['fastestLapTime'].min().reset_index()
fastest_lap_per_race = fastest_lap_per_race[['raceId', 'fastestLapTime']]
fastest_lap_per_race = fastest_lap_per_race.rename(columns={'fastestLapTime': 'Race Fastest Lap'})

In [39]:
fastest_lap_per_race.head()

,raceId,Race Fastest Lap
0,989,1:25.9
1,990,1:33.7
2,991,1:35.8
3,992,1:45.1
4,993,1:18.4


In [28]:
seconds = 90

In [29]:
minutes = seconds // 60
seconds_remaining = seconds % 60
print(f'{int(minutes)}:{seconds_remaining:.1f}')

1:30.0


In [30]:
with open('../results/stats_2018.json', 'r') as f:
        json_data = json.load(f)

In [31]:
json_data[0]['Race Name']

'Abu Dhabi Grand Prix'

In [32]:
def from_seconds_to_original_fastest_lap_time(fastest_lap_time):
    if pd.isna(fastest_lap_time):
        return None
    minutes = fastest_lap_time // 60
    seconds = fastest_lap_time % 60
    return f'{int(minutes)}:{seconds:.1f}'

In [33]:
fastest_lap_per_race['Race Fastest Lap'] = fastest_lap_per_race['Race Fastest Lap'].apply(from_seconds_to_original_fastest_lap_time)

In [34]:
race_analytics_output = joined_tables.merge(winner_driver, on='raceId', how='left') 
race_analytics_output = race_analytics_output.merge(fastest_lap_per_race, on='raceId', how='left') 

In [35]:
race_analytics_cols_to_keep = race_analytics_output[[
                            'name', 
                            'year',
                            'round',
                            'datetime',
                            'Race Winning driverId',
                            'Race Fastest Lap']
                        ].rename(columns={
                            'name': 'Race Name', 
                            'round': 'Race Round',
                            'datetime': 'Race Datetime',
                        })

In [36]:
# the json value for a key is always a number, represent it as such rather than a string

race_analytics_cols_to_keep[race_analytics_cols_to_keep['Race Winning driverId'].isna()] = None
race_analytics_cols_to_keep['Race Winning driverId'] = race_analytics_cols_to_keep['Race Winning driverId'].astype('Int64')
race_analytics_cols_to_keep['Race Round'] = race_analytics_cols_to_keep['Race Round'].astype('Int64')

In [37]:
# make sure the year is an integer, not a float, so that we do not have json files as such: stats_2000.0.json, stats_2001.0.json

race_analytics_cols_to_keep['year'] = race_analytics_cols_to_keep['year'].astype('Int64')
data_grouped_by_year = race_analytics_cols_to_keep.groupby('year')

In [38]:
for year, data in data_grouped_by_year:
    data = data.drop(columns=['year'])
    json_output = data.to_dict(orient='records')
    with open(f'../results/stats_{year}.json', 'w') as fd:
        json.dump(json_output, fd, indent=2)